# Volatility Forecasting on Equity Returns: GARCH-Family Models vs. Gradient Boosting

**Author:** Raffaele Perillo

This notebook builds a conditional-mean/conditional-variance pipeline on daily equity returns, benchmarks GARCH, EGARCH and Asymmetric Power ARCH specifications using both in-sample fit statistics and genuine out-of-sample forecast-validation tests, and compares the winning econometric model against a Gradient Boosting model trained on realized market variance, excess returns and short-term rate features.

**Key finding:** the model with the best in-sample fit (EGARCH/APARCH, by HQIC and log-likelihood) is *not* the model that survives an out-of-sample unbiasedness/efficiency test on its forecasts — GARCH(1,1) is. The Gradient Boosting benchmark then slightly outperforms GARCH(1,1) on out-of-sample MAE, but produces visibly smoother forecasts that miss most volatility spikes, illustrating why a single loss metric can be misleading for tail-risk-sensitive applications.


In [ ]:
!pip install arch

In [ ]:
import itertools
import math
import os
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import scipy.stats as stats
from scipy import stats as scipy_stats
from scipy.stats import jarque_bera
from scipy.optimize import minimize
import statsmodels.api as sm
from statsmodels.tsa.api import ExponentialSmoothing, Holt, SimpleExpSmoothing
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.stattools import acf, pacf, q_stat, adfuller, kpss
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from arch import arch_model
from arch.univariate import APARCH, Normal, ZeroMean, EWMAVariance, ConstantMean
from arch.unitroot import PhillipsPerron
import yfinance as yf
from sklearn import tree
from sklearn.datasets import make_classification
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import accuracy_score, mean_squared_error
from sklearn.model_selection import TimeSeriesSplit, train_test_split, cross_val_score
from sklearn.neural_network import MLPClassifier, MLPRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor, export_graphviz
from lightgbm import LGBMRegressor
from xgboost import XGBRFRegressor
from graphviz import Source

***Set default layout.***

In [ ]:
 default_layout = dict(
    xaxis=dict(titlefont=dict(size=25), tickfont=dict(size=20), showgrid=False),
    yaxis=dict(titlefont=dict(size=25), tickfont=dict(size=20), showgrid=False, zeroline=False),
    font=dict(family="Serif", size=20, color="black"),
    plot_bgcolor='rgba(0,0,0,0)',
    paper_bgcolor='rgba(0,0,0,0)',
    legend=dict(font=dict(size=18), bgcolor='rgba(255,255,255,0.5)', bordercolor='black', borderwidth=1),
    margin=dict(t=80, l=60, r=60, b=60)
)

## Data Exploration

In [ ]:
prices_df = pd.read_excel('/content/prices2.xlsx', sheet_name='prices')
prices_df['date'] = pd.to_datetime(prices_df['date'])
prices_df.set_index('date', inplace = True)
prices_df = prices_df.loc['22/01/2001':'29/04/2025']

In [ ]:
returns_df = pd.DataFrame()
returns_df["F US Equity Returns"] = prices_df[['F US Equity']].pct_change()
returns_df.dropna(inplace = True)

In [ ]:
returns_df

,F US Equity Returns
date,
2001-01-23,-0.004608
2001-01-24,0.027778
2001-01-25,0.011261
2001-01-26,-0.002227
2001-01-29,0.010357
...,...
2025-04-23,0.013472
2025-04-24,0.028630
2025-04-25,-0.001988


## Shortcut functions

In [ ]:
def SACF_SPACF(series, lag_max = 24, alpha_level = 0.05, model_df = 0):
    #takes as mandatory argument the series, 24 is the numeber of lags
    """
    Compute the sample autocorrelation function (SACF), sample partial autocorrelation function (SPACF),
    and Ljung-Box Q-statistics for the SACF.

    This function calculates the ACF and PACF values along with their corresponding confidence intervals
    for lags 1 through `lag_max` using the provided significance level (`alpha_level`). In addition, it
    computes the Ljung-Box Q-statistic and associated p-values (excluding lag 0) for the SACF.

    Set `model_df` to the number of dof lost (= number of conditional mean coefficents estimated,
    i.e. all coeffs excluding the intercept).

    """

    # Calculate ACF and PACF with confidence intervals
    acf_vals, acf_confint = acf(series, nlags=lag_max, alpha=alpha_level)
    pacf_vals, pacf_confint = pacf(series, nlags=lag_max, alpha=alpha_level, method='ols')

    # Calculate Ljung-Box statistics and p-values
    lb_results = sm.stats.acorr_ljungbox(
        series,
        lags=range(1, lag_max + 1),
        model_df=model_df,
        return_df=True
    )

    # Build the results DataFrame
    df_acf_pacf = pd.DataFrame({
        "Lag": np.arange(1, lag_max + 1),
        "ACF": acf_vals[1:],
        "ACF_lower": acf_confint[1:, 0],
        "ACF_upper": acf_confint[1:, 1],
        "PACF": pacf_vals[1:],
        "PACF_lower": pacf_confint[1:, 0],
        "PACF_upper": pacf_confint[1:, 1],
        "Q-stat": lb_results["lb_stat"].values,
        "Q-stat Prob": lb_results["lb_pvalue"].values.round(6)
    })

    # Set the index to 'Lag' and extract the main columns
    df_acf_pacf.set_index("Lag", inplace=True)
    df_acf_pacf_small = df_acf_pacf[["ACF", "PACF", "Q-stat", "Q-stat Prob"]].copy()

    return df_acf_pacf_small

In [ ]:
def SACF_SPACF_plotly(series, lag_max=24, ylim=(-0.15, 0.15)):
    # Compute ACF and PACF with confidence intervals
    acf_vals, acf_conf = acf(series, nlags=lag_max, alpha=0.05)
    pacf_vals, pacf_conf = pacf(series, nlags=lag_max, alpha=0.05, method='ols')

    lags = list(range(1, lag_max+1))

    fig = make_subplots(rows=1, cols=2, subplot_titles=("SACF", "SPACF"))

    # ACF: bars + CIs
    fig.add_trace(go.Bar(
        x=lags, y=acf_vals[1:], marker_color='blue', opacity=0.7,
        showlegend=False
    ), row=1, col=1)

    fig.add_trace(go.Scatter(
        x=lags, y=acf_conf[1:,0], mode='lines',
        line=dict(color='grey', width=2),
        showlegend=False
    ), row=1, col=1)

    fig.add_trace(go.Scatter(
        x=lags, y=acf_conf[1:,1], mode='lines',
        line=dict(color='grey', width=2),
        fill='tonexty', fillcolor='rgba(128,128,128,0.15)',
        showlegend=False
    ), row=1, col=1)

    # PACF: bars + CIs
    fig.add_trace(go.Bar(
        x=lags, y=pacf_vals[1:], marker_color='blue', opacity=0.7,
        showlegend=False
    ), row=1, col=2)

    fig.add_trace(go.Scatter(
        x=lags, y=pacf_conf[1:,0], mode='lines',
        line=dict(color='grey', width=2),
        showlegend=False
    ), row=1, col=2)

    fig.add_trace(go.Scatter(
        x=lags, y=pacf_conf[1:,1], mode='lines',
        line=dict(color='grey', width=2),
        fill='tonexty', fillcolor='rgba(128,128,128,0.15)',
        showlegend=False
    ), row=1, col=2)

    # Layout
    fig.update_yaxes(range=ylim, row=1, col=1)
    fig.update_yaxes(range=ylim, row=1, col=2)

    fig.update_layout(
        title=dict(text='SACF and SPACF', x=0.5, font=dict(size=28, family='Serif', color='black')),
        font=dict(family="Serif", size=18, color="black"),
        plot_bgcolor='rgba(0,0,0,0)',
        paper_bgcolor='rgba(0,0,0,0)',
        margin=dict(t=80, l=60, r=60, b=60),
        showlegend=False
    )

    fig.show()

## 1. Conditional Mean Model: ARMA(2,1)

ARMA (2,1)

In [ ]:
model = sm.tsa.statespace.SARIMAX(
    returns_df,
    order=(2, 0, 1),
    trend='c',
    enforce_stationarity=False,
    enforce_invertibility=False
)
cond_mean_model = model.fit()
print(cond_mean_model.summary())

/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)


                                SARIMAX Results                                
Dep. Variable:     F US Equity Returns   No. Observations:                 6103
Model:                SARIMAX(2, 0, 1)   Log Likelihood               13523.885
Date:                 Sat, 16 May 2026   AIC                         -27037.770
Time:                         19:14:55   BIC                         -27004.189
Sample:                              0   HQIC                        -27026.118
                                - 6103                                         
Covariance Type:                   opg                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
intercept   9.809e-05      0.000      0.446      0.656      -0.000       0.001
ar.L1          0.3892      0.138      2.824      0.005       0.119       0.659
ar.L2          0.0326      0.010      3.339 

L-BFGS converged successfully

## Extracting Residuals from the Conditional Mean Model

In [ ]:
residuals_df = pd.DataFrame()
residuals_df['Residuals'] = cond_mean_model.resid

## 2. Baseline Conditional Variance Model: GARCH(1,1)

###***GARCH(1,1) before rescaling***

In [ ]:
garch = arch_model(residuals_df, mean='Zero', vol='Garch', p=1, q=1, dist='normal')
garch_1_1 = garch.fit()
predicted_garch_volatility = garch_1_1.conditional_volatility
predicted_garch_variance = predicted_garch_volatility**2
print(garch_1_1.summary())

Iteration:      1,   Func. Count:      4,   Neg. LLF: -14595.519645942302
Optimization terminated successfully    (Exit mode 0)
            Current function value: -14595.519636292724
            Iterations: 5
            Function evaluations: 4
            Gradient evaluations: 1
                       Zero Mean - GARCH Model Results                        
Dep. Variable:              Residuals   R-squared:                       0.000
Mean Model:                 Zero Mean   Adj. R-squared:                  0.000
Vol Model:                      GARCH   Log-Likelihood:                14595.5
Distribution:                  Normal   AIC:                          -29185.0
Method:            Maximum Likelihood   BIC:                          -29164.9
                                        No. Observations:                 6103
Date:                Sat, May 16 2026   Df Residuals:                     6103
Time:                        19:14:56   Df Model:                            0
       

/usr/local/lib/python3.12/dist-packages/arch/univariate/base.py:694: DataScaleWarning: y is poorly scaled, which may affect convergence of the optimizer when
estimating the model parameters. The scale of y is 0.0006952. Parameter
estimation work better when this value is between 1 and 1000. The recommended
rescaling is 100 * y.

This warning can be disabled by either rescaling y before initializing the
model or by setting rescale=False.

  self._check_scale(resids)


###***GARCH(1,1) after rescaling***

In [ ]:
residuals_scaled_df = residuals_df * 100

In [ ]:
garch = arch_model(residuals_scaled_df, mean='Zero', vol='Garch', p=1, q=1, dist='normal')
garch_1_1 = garch.fit()
epsilons_garch_scaled = garch_1_1.resid
epsilons_garch_squared_scaled = epsilons_garch_scaled**2
predicted_garch_volatility_scaled = garch_1_1.conditional_volatility
predicted_garch_variance_scaled = predicted_garch_volatility_scaled**2
print(garch_1_1.summary())

Iteration:      1,   Func. Count:      5,   Neg. LLF: 42101.42924431918
Iteration:      2,   Func. Count:     13,   Neg. LLF: 1109596655.241461
Iteration:      3,   Func. Count:     18,   Neg. LLF: 13594.225457877548
Iteration:      4,   Func. Count:     23,   Neg. LLF: 13538.759901488438
Iteration:      5,   Func. Count:     28,   Neg. LLF: 21448.141149435112
Iteration:      6,   Func. Count:     34,   Neg. LLF: 13544.260424546788
Iteration:      7,   Func. Count:     39,   Neg. LLF: 13507.545718446312
Iteration:      8,   Func. Count:     44,   Neg. LLF: 13504.51319270159
Iteration:      9,   Func. Count:     49,   Neg. LLF: 13503.395371399572
Iteration:     10,   Func. Count:     53,   Neg. LLF: 13503.395263588201
Iteration:     11,   Func. Count:     57,   Neg. LLF: 13503.395263064234
Optimization terminated successfully    (Exit mode 0)
            Current function value: 13503.395263064234
            Iterations: 11
            Function evaluations: 57
            Gradient evalua

## Stationarity and Positivity Checks

*Parameter Extraction:*

In [ ]:
omega_garch_scaled = garch_1_1.params['omega']
alpha_garch = garch_1_1.params['alpha[1]']
beta_garch = garch_1_1.params['beta[1]']

For stationarity, the condition  𝛼+𝛽<1  must hold

In [ ]:
sum = alpha_garch + beta_garch
sum < 1

np.True_

In [ ]:
sum

np.float64(0.9886922807546064)

The stationarity condition is satisfied

## In-Sample Fit: Predicted Conditional Variance vs. Squared Residuals

***Rescaling is required:***

In [ ]:
comparison_residuals_df = pd.DataFrame({
    'epsilons_garch_scaled': garch_1_1.resid,
    'residuals': residuals_df ['Residuals']
})
comparison_residuals_df.head()

,epsilons_garch_scaled,residuals
date,,
2001-01-23,-0.460829,-0.004608
2001-01-24,2.947334,0.029473
2001-01-25,0.050174,0.000502
2001-01-26,-0.745448,-0.007454
2001-01-29,0.811202,0.008112


In [ ]:
epsilons_garch = epsilons_garch_scaled/100
epsilons_garch_squared = epsilons_garch_squared_scaled/(100**2)

In [ ]:
comparison_residuals_df = pd.DataFrame({
    'epsilons_garch': epsilons_garch,
    'residuals': residuals_df ['Residuals']
})
comparison_residuals_df.head()

,epsilons_garch,residuals
date,,
2001-01-23,-0.004608,-0.004608
2001-01-24,0.029473,0.029473
2001-01-25,0.000502,0.000502
2001-01-26,-0.007454,-0.007454
2001-01-29,0.008112,0.008112


In [ ]:
predicted_garch_volatility = predicted_garch_volatility_scaled/100
predicted_garch_variance = predicted_garch_variance_scaled/(100**2)

In [ ]:
fig_garch = go.Figure()

fig_garch.add_trace(go.Scatter(
    x=returns_df.index,
    y=epsilons_garch_squared,
    mode='lines',
    name='Squared Residual',
    line=dict(color='steelblue', width=1.5),
    opacity=0.5
))

fig_garch.add_trace(go.Scatter(
    x=returns_df.index,
    y=predicted_garch_variance,
    mode='lines',
    name=f'GARCH(1,1) Variance',
    line=dict(color='firebrick', width=2.0)
))

fig_garch.update_layout(
    title=dict(
        text=f'GARCH(1,1) Predicted Variance',
        x=0.5,
        font=dict(size=30, family='Serif', color='black')
    ),
    xaxis=dict(
        title='Date',
        titlefont=dict(size=25),
        tickfont=dict(size=20),
        showgrid=False
    ),
    yaxis=dict(
        title='Squared Residual / Variance',
        titlefont=dict(size=25),
        tickfont=dict(size=20),
        showgrid=False,
        zeroline=False,
    ),
    font=dict(
        family="Serif",
        size=20,
        color="black"
    ),

    plot_bgcolor='rgba(0,0,0,0)',

    paper_bgcolor='rgba(0,0,0,0)',

    legend=dict(
        font=dict(size=18),
        bgcolor='rgba(255,255,255,0.5)',
        bordercolor='black',
        borderwidth=1
    ),
    margin=dict(t=80, l=60, r=60, b=60)
)

fig_garch.show()

## Diagnostics: Correlograms of Squared Standardized Residuals (SACF/SPACF)

In [ ]:
standardized_residuals_garch_1_1 = epsilons_garch/predicted_garch_volatility
squared_standardized_residuals_garch_1_1 = standardized_residuals_garch_1_1**2

In [ ]:
SACF_SPACF_plotly(squared_standardized_residuals_garch_1_1,lag_max=12, ylim=[-0.1,0.1])

In [ ]:
SACF_SPACF_plotly(squared_standardized_residuals_garch_1_1,lag_max=22, ylim=[-0.1,0.1])

## ARCH-LM Test for Remaining Conditional Heteroskedasticity


We performed the ARCH-LM auxiliary regression on both the squared standardized residuals from the conditional mean model and from the GARCH(1,1) model. However, we based our analysis and model selection on the former.

### ***ARCH_LM test on Raw data***

In [ ]:
standardized_residuals_cond_mean_model = residuals_df/residuals_df.std()
squared_standardized_residuals_cond_mean_model = (standardized_residuals_cond_mean_model)**2

In [ ]:
X = pd.DataFrame()
for lag in range(1, 13):
    X[f'y_lagged_{lag}'] = standardized_residuals_cond_mean_model.shift(lag)
    X[f'sqr_y_lagged_{lag}'] = squared_standardized_residuals_cond_mean_model.shift(lag)
X.dropna(inplace=True)
X_with_constant = sm.add_constant(X)
y = squared_standardized_residuals_cond_mean_model.copy()
y = y.iloc[12:]
auxiliary_regression = sm.OLS(y, X_with_constant)
results_auxiliary_regression = auxiliary_regression.fit()
print(results_auxiliary_regression.summary())

                            OLS Regression Results                            
Dep. Variable:              Residuals   R-squared:                       0.181
Model:                            OLS   Adj. R-squared:                  0.178
Method:                 Least Squares   F-statistic:                     56.04
Date:                Sat, 16 May 2026   Prob (F-statistic):          5.99e-242
Time:                        19:15:02   Log-Likelihood:                -16496.
No. Observations:                6091   AIC:                         3.304e+04
Df Residuals:                    6066   BIC:                         3.321e+04
Df Model:                          24                                         
Covariance Type:            nonrobust                                         
                      coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------
const               0.3117      0.052     

In [ ]:
T = len(y)
R2 = results_auxiliary_regression.rsquared
q = 24
LM_stat = T * R2
p_value = 1 - stats.chi2.cdf(LM_stat, df=q)
print(f"LM_stat:                        {LM_stat:.4f}")
print(f"p_value:                        {p_value:.5f}")

LM_stat:                        1105.4093
p_value:                        0.00000


### ***ARCH_LM test after the GARCH_1_1***

In [ ]:
X = pd.DataFrame()
for lag in range(1, 13):
    X[f'y_lagged_{lag}'] = standardized_residuals_garch_1_1.shift(lag)
    X[f'sqr_y_lagged_{lag}'] = squared_standardized_residuals_garch_1_1.shift(lag)
X.dropna(inplace=True)
X_with_constant = sm.add_constant(X)
y = squared_standardized_residuals_garch_1_1.copy()
y = y.iloc[12:]
auxiliary_regression = sm.OLS(y, X_with_constant)
results_auxiliary_regression = auxiliary_regression.fit()
print(results_auxiliary_regression.summary())

                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.003
Model:                            OLS   Adj. R-squared:                 -0.001
Method:                 Least Squares   F-statistic:                    0.8248
Date:                Sat, 16 May 2026   Prob (F-statistic):              0.708
Time:                        19:15:02   Log-Likelihood:                -14663.
No. Observations:                6091   AIC:                         2.938e+04
Df Residuals:                    6066   BIC:                         2.954e+04
Df Model:                          24                                         
Covariance Type:            nonrobust                                         
                      coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------
const               1.0490      0.057     

In [ ]:
T = len(y)
R2 = results_auxiliary_regression.rsquared
q = 24
LM_stat = T * R2
p_value = 1 - stats.chi2.cdf(LM_stat, df=q)
print(f"LM_stat:                        {LM_stat:.4f}")
print(f"p_value:                        {p_value:.5f}")

LM_stat:                        19.8112
p_value:                        0.70746


## 3. Addressing the Leverage Effect: Selecting an Alternative Conditional Variance Model (EGARCH)

In [ ]:
egarch = arch_model(
    residuals_scaled_df,
    mean='Zero',
    vol='EGARCH',
    p=1,
    o=1,
    q=1,
    dist='normal'
)
arch_lm_model = egarch.fit()
epsilons_egarch_scaled = arch_lm_model.resid
epsilons_egarch_squared_scaled = epsilons_egarch_scaled**2
predicted_egarch_volatility_scaled = arch_lm_model.conditional_volatility
predicted_egarch_variance_scaled = predicted_egarch_volatility_scaled**2
print(arch_lm_model.summary())

Iteration:      1,   Func. Count:      6,   Neg. LLF: 52202.42994027698
Iteration:      2,   Func. Count:     17,   Neg. LLF: 52202.31423205714
Iteration:      3,   Func. Count:     27,   Neg. LLF: 717518307.8300931
Iteration:      4,   Func. Count:     36,   Neg. LLF: 13519.098650531003
Iteration:      5,   Func. Count:     42,   Neg. LLF: 1488428024.767868
Iteration:      6,   Func. Count:     50,   Neg. LLF: 13464.767651687056
Iteration:      7,   Func. Count:     55,   Neg. LLF: 13464.676989462212
Iteration:      8,   Func. Count:     60,   Neg. LLF: 13464.667106801375
Iteration:      9,   Func. Count:     65,   Neg. LLF: 13464.667014971703
Iteration:     10,   Func. Count:     70,   Neg. LLF: 13464.667013493548
Iteration:     11,   Func. Count:     74,   Neg. LLF: 13464.667013492573
Optimization terminated successfully    (Exit mode 0)
            Current function value: 13464.667013493548
            Iterations: 11
            Function evaluations: 74
            Gradient evaluat

***Rescaling***

In [ ]:
epsilons_egarch = epsilons_egarch_scaled/100
epsilons_egarch_squared = epsilons_egarch_squared_scaled/(100**2)
predicted_egarch_volatility = predicted_egarch_volatility_scaled/100
predicted_egarch_variance = predicted_egarch_variance_scaled/(100**2)

## EGARCH(1,1) Estimation Output

In [ ]:
fig_garch = go.Figure()

fig_garch.add_trace(go.Scatter(
    x=returns_df.index,
    y=epsilons_egarch_squared,
    mode='lines',
    name='Squared Residual',
    line=dict(color='steelblue', width=1.5),
    opacity=0.5
))

fig_garch.add_trace(go.Scatter(
    x=returns_df.index,
    y=predicted_egarch_variance,
    mode='lines',
    name=f'Arch_lm_model Variance',
    line=dict(color='firebrick', width=2.0)
))

fig_garch.update_layout(
    title=dict(
        text=f'Arch_lm_model Predicted Variance',
        x=0.5,
        font=dict(size=30, family='Serif', color='black')
    ),
    xaxis=dict(
        title='Date',
        titlefont=dict(size=25),
        tickfont=dict(size=20),
        showgrid=False
    ),
    yaxis=dict(
        title='Squared Residual / Variance',
        titlefont=dict(size=25),
        tickfont=dict(size=20),
        showgrid=False,
        zeroline=False,
    ),
    font=dict(
        family="Serif",
        size=20,
        color="black"
    ),

    plot_bgcolor='rgba(0,0,0,0)',

    paper_bgcolor='rgba(0,0,0,0)',

    legend=dict(
        font=dict(size=18),
        bgcolor='rgba(255,255,255,0.5)',
        bordercolor='black',
        borderwidth=1
    ),
    margin=dict(t=80, l=60, r=60, b=60)
)

fig_garch.show()

## In-Sample Fit: EGARCH(1,1) vs. Squared Residuals

In [ ]:
Log_Likelihood_arch_lm_model = -13464.7
HQIC_arch_lm_model = (-2*Log_Likelihood_arch_lm_model)+np.log((np.log(6103)))*2*4

In [ ]:
Log_Likelihood_garch_1_1 = -13503.4
HQIC_garch_1_1 = (-2*Log_Likelihood_garch_1_1)+np.log((np.log(6103)))*2*3

In [ ]:
# @title
comparison_table = pd.DataFrame({
    'Model': ['arch_lm_model', 'Garch_1_1'],
    'Log-Likelihood': [Log_Likelihood_arch_lm_model, Log_Likelihood_garch_1_1],
    '      k (params)': [4, 3],
    'HQIC': [HQIC_arch_lm_model, HQIC_garch_1_1]
})

comparison_table['Best HQIC'] = comparison_table['HQIC'] == comparison_table['HQIC'].min()
comparison_table['Best HQIC'] = comparison_table['Best HQIC'].map({True: '✓', False: ''})

print(comparison_table.to_string(index=False))

        Model  Log-Likelihood        k (params)         HQIC Best HQIC
arch_lm_model        -13464.7                 4 26946.721775         ✓
    Garch_1_1        -13503.4                 3 27019.791331          


## Model Comparison: GARCH(1,1) vs. EGARCH(1,1) (HQIC & Log-Likelihood)

In [ ]:
aparch = arch_model(residuals_scaled_df, mean='Zero', vol='APARCH', p=2, o=2, q=4, dist='normal')
adv_cond_var_model = aparch.fit()
epsilons_aparch_scaled = adv_cond_var_model.resid
epsilons_aparch_squared_scaled = epsilons_aparch_scaled**2
predicted_aparch_volatility_scaled = adv_cond_var_model.conditional_volatility
predicted_aparch_variance_scaled = predicted_aparch_volatility_scaled**2
print(adv_cond_var_model.summary())

Iteration:      1,   Func. Count:     12,   Neg. LLF: 52203.64150825611
Iteration:      2,   Func. Count:     28,   Neg. LLF: 49050.29868923387
Iteration:      3,   Func. Count:     40,   Neg. LLF: 51437.11800849797
Iteration:      4,   Func. Count:     52,   Neg. LLF: 52057.016266016624
Iteration:      5,   Func. Count:     64,   Neg. LLF: 13744.388264348949
Iteration:      6,   Func. Count:     76,   Neg. LLF: 28935.982702274174
Iteration:      7,   Func. Count:     88,   Neg. LLF: 289103.2301116127
Iteration:      8,   Func. Count:    100,   Neg. LLF: 18696.741973730714
Iteration:      9,   Func. Count:    112,   Neg. LLF: 23315.031774601364
Iteration:     10,   Func. Count:    124,   Neg. LLF: 37774.378456546314
Iteration:     11,   Func. Count:    136,   Neg. LLF: 19466.38910037041
Iteration:     12,   Func. Count:    148,   Neg. LLF: 14522.818387589428
Iteration:     13,   Func. Count:    160,   Neg. LLF: 13678.538992525024
Iteration:     14,   Func. Count:    172,   Neg. LLF: 17

***Rescaling***

In [ ]:
epsilons_aparch = epsilons_aparch_scaled/100
epsilons_aparch_squared = epsilons_aparch_squared_scaled/(100**2)
predicted_aparch_volatility = predicted_aparch_volatility_scaled/100
predicted_aparch_variance = predicted_aparch_variance_scaled/(100**2)

## Which Model Wins on Information Criteria?

In [ ]:
Log_Likelihood_adv_cond_var_model = -13425.0
HQIC_adv_cond_var_model = (-2*Log_Likelihood_adv_cond_var_model)+np.log((np.log(6103)))*2*10

In [ ]:
comparison_table = pd.DataFrame({
    'Model': ['arch_lm_model', 'adv_cond_var_model'],
    'Log-Likelihood': [Log_Likelihood_arch_lm_model, Log_Likelihood_adv_cond_var_model],
    'k (params)': [4, 10],
    'HQIC': [HQIC_arch_lm_model, HQIC_adv_cond_var_model]
})

comparison_table['Best HQIC'] = comparison_table['HQIC'] == comparison_table['HQIC'].min()
comparison_table['Best HQIC'] = comparison_table['Best HQIC'].map({True: '✓', False: ''})

print(comparison_table.to_string(index=False))

             Model  Log-Likelihood  k (params)         HQIC Best HQIC
     arch_lm_model        -13464.7           4 26946.721775          
adv_cond_var_model        -13425.0          10 26893.304438         ✓


## 4. A More Flexible Alternative: Asymmetric Power ARCH(2,4)

In [ ]:
squared_residuals = residuals_df['Residuals']**2

In [ ]:
MAE_arch_lm_model = np.mean(np.abs( squared_residuals - predicted_egarch_variance))
MAE_adv_cond_var_model = np.mean(np.abs( squared_residuals - predicted_aparch_variance))
MAE_garch_1_1 = np.mean(np.abs( squared_residuals - predicted_garch_variance))

In [ ]:
print(f"MAE_garch_1_1:                        {MAE_garch_1_1:.4f}")
print(f"MAE_arch_lm_model:                        {MAE_arch_lm_model:.6f}")
print(f"MAE_adv_cond_var_model:                       {MAE_adv_cond_var_model:.6f}")

MAE_garch_1_1:                        0.0008
MAE_arch_lm_model:                        0.000749
MAE_adv_cond_var_model:                       0.000742


## Comparing In-Sample MAE Across Candidate Models

In [ ]:
pred_df = pd.DataFrame()
pred_df['epsilon_squared']=residuals_df['Residuals']**2
pred_df = pd.concat([pred_df['epsilon_squared'] , predicted_garch_variance, predicted_egarch_variance, predicted_aparch_variance,], axis=1, join='inner')
pred_df.columns = ['Epsilon Squared','Predicted Variance GARCH', 'Predicted Variance Arch_lm_model','Predicted Variance adv_cond_var_model']

In [ ]:
Y = pred_df['Epsilon Squared']
test = dict()

for col in pred_df.columns:
    if col != 'Epsilon Squared':
        X = pred_df[col]
        X = sm.add_constant(X)
        model = sm.OLS(Y, X)
        results = model.fit()

        test[col] = {
            'summary': results.summary().as_text(),
            'f_test': None,
            't_test': None
        }

        # Perform and store the F-test
        try:
            f_test = results.f_test(f"const = 0, {col} = 1")
            test[col]['f_test'] = f_test
        except Exception as e:
            test[col]['f_test'] = f"Error performing F-test: {e}"

        # Perform and store the T-test
        try:
            t_test = results.t_test(f"{col} = 1")
            test[col]['t_test'] = t_test
        except Exception as e:
            test[col]['t_test'] = f"Error performing T-test: {e}"

###***Garch (1,1) - Winner***

In [ ]:
column_name = 'Predicted Variance GARCH'
if column_name in test:
    print(f"Regression results for {column_name}:")
    print(test[column_name]['summary'])
    print("\nF-test for hypothesis that intercept = 0 and coefficient of regressor = 1:")
    print(test[column_name]['f_test'])
    print("\nT-test for hypothesis that the coefficient of regressor = 1:")
    print(test[column_name]['t_test'])
else:
    print(f"No results available for {column_name}")

Regression results for Predicted Variance GARCH:
                            OLS Regression Results                            
Dep. Variable:        Epsilon Squared   R-squared:                       0.121
Model:                            OLS   Adj. R-squared:                  0.121
Method:                 Least Squares   F-statistic:                     839.2
Date:                Sat, 16 May 2026   Prob (F-statistic):          5.35e-173
Time:                        19:15:08   Log-Likelihood:                 27636.
No. Observations:                6103   AIC:                        -5.527e+04
Df Residuals:                    6101   BIC:                        -5.525e+04
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                               coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------

###***Arch_lm_model***

In [ ]:
column_name = 'Predicted Variance Arch_lm_model'
if column_name in test:
    print(f"Regression results for {column_name}:")
    print(test[column_name]['summary'])
    print("\nF-test for hypothesis that intercept = 0 and coefficient of regressor = 1:")
    print(test[column_name]['f_test'])
    print("\nT-test for hypothesis that the coefficient of regressor = 1:")
    print(test[column_name]['t_test'])
else:
    print(f"No results available for {column_name}")

Regression results for Predicted Variance Arch_lm_model:
                            OLS Regression Results                            
Dep. Variable:        Epsilon Squared   R-squared:                       0.130
Model:                            OLS   Adj. R-squared:                  0.130
Method:                 Least Squares   F-statistic:                     908.8
Date:                Sat, 16 May 2026   Prob (F-statistic):          3.11e-186
Time:                        19:15:08   Log-Likelihood:                 27666.
No. Observations:                6103   AIC:                        -5.533e+04
Df Residuals:                    6101   BIC:                        -5.531e+04
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                                       coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------

###***Adv_cond_var_model***

In [ ]:
column_name = 'Predicted Variance adv_cond_var_model'
if column_name in test:
    print(f"Regression results for {column_name}:")
    print(test[column_name]['summary'])
    print("\nF-test for hypothesis that intercept = 0 and coefficient of regressor = 1:")
    print(test[column_name]['f_test'])
    print("\nT-test for hypothesis that the coefficient of regressor = 1:")
    print(test[column_name]['t_test'])
else:
    print(f"No results available for {column_name}")

Regression results for Predicted Variance adv_cond_var_model:
                            OLS Regression Results                            
Dep. Variable:        Epsilon Squared   R-squared:                       0.135
Model:                            OLS   Adj. R-squared:                  0.135
Method:                 Least Squares   F-statistic:                     949.4
Date:                Sat, 16 May 2026   Prob (F-statistic):          6.71e-194
Time:                        19:15:08   Log-Likelihood:                 27684.
No. Observations:                6103   AIC:                        -5.536e+04
Df Residuals:                    6101   BIC:                        -5.535e+04
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                                            coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------

## 5. Forecast Validation: Testing Unbiasedness and Efficiency Out-of-Sample

In [ ]:
garch = arch_model(residuals_scaled_df, mean='Zero', vol='Garch', p=1, q=1, dist='normal')
winning_cond_var_model = garch.fit()
print(winning_cond_var_model.summary())

Iteration:      1,   Func. Count:      5,   Neg. LLF: 42101.42924431918
Iteration:      2,   Func. Count:     13,   Neg. LLF: 1109596655.241461
Iteration:      3,   Func. Count:     18,   Neg. LLF: 13594.225457877548
Iteration:      4,   Func. Count:     23,   Neg. LLF: 13538.759901488438
Iteration:      5,   Func. Count:     28,   Neg. LLF: 21448.141149435112
Iteration:      6,   Func. Count:     34,   Neg. LLF: 13544.260424546788
Iteration:      7,   Func. Count:     39,   Neg. LLF: 13507.545718446312
Iteration:      8,   Func. Count:     44,   Neg. LLF: 13504.51319270159
Iteration:      9,   Func. Count:     49,   Neg. LLF: 13503.395371399572
Iteration:     10,   Func. Count:     53,   Neg. LLF: 13503.395263588201
Iteration:     11,   Func. Count:     57,   Neg. LLF: 13503.395263064234
Optimization terminated successfully    (Exit mode 0)
            Current function value: 13503.395263064234
            Iterations: 11
            Function evaluations: 57
            Gradient evalua

## Selecting the Best Model on Forecast-Validation Grounds

In [ ]:
df_forecast = returns_df.copy()

In [ ]:
train_size = int(len(df_forecast['F US Equity Returns']) * 0.80)
train_val = df_forecast['F US Equity Returns'].iloc[:train_size]
test = df_forecast['F US Equity Returns'].iloc[train_size:]
test_size = len(test)

## Train/Validation/Test Split for Genuine Out-of-Sample Forecasting

In [ ]:
model = sm.tsa.statespace.SARIMAX(
    train_val,
    order=(2, 0, 1),
    trend='c',
    enforce_stationarity=False,
    enforce_invertibility=False
)
cond_mean_model_train_val = model.fit()
print(cond_mean_model_train_val.summary())

/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning:

A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.

/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning:

A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.



                                SARIMAX Results                                
Dep. Variable:     F US Equity Returns   No. Observations:                 4882
Model:                SARIMAX(2, 0, 1)   Log Likelihood               10775.580
Date:                 Sat, 16 May 2026   AIC                         -21541.159
Time:                         19:15:12   BIC                         -21508.695
Sample:                              0   HQIC                        -21529.767
                                - 4882                                         
Covariance Type:                   opg                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
intercept   1.416e-05      0.000      0.076      0.939      -0.000       0.000
ar.L1          0.5601      0.084      6.640      0.000       0.395       0.725
ar.L2          0.0306      0.010      3.170 

In [ ]:
train_val_residuals = cond_mean_model_train_val.resid
train_val_residuals_scaled = train_val_residuals * 100

In [ ]:
garch = arch_model(train_val_residuals_scaled, mean='Zero', vol='Garch', p=1, q=1, dist='normal')
garch_result = garch.fit()
predicted_garch_volatility_scaled_train = garch_result.conditional_volatility
predicted_garch_variance_scaled_train = predicted_garch_volatility_scaled_train**2
print(garch_result.summary())

Iteration:      1,   Func. Count:      5,   Neg. LLF: 2832058393.821159
Iteration:      2,   Func. Count:     12,   Neg. LLF: 10666.00570597796
Iteration:      3,   Func. Count:     17,   Neg. LLF: 798483568.1783601
Iteration:      4,   Func. Count:     22,   Neg. LLF: 10626.57278590766
Iteration:      5,   Func. Count:     27,   Neg. LLF: 10608.885600334976
Iteration:      6,   Func. Count:     32,   Neg. LLF: 10672.372169401526
Iteration:      7,   Func. Count:     37,   Neg. LLF: 10595.506198941457
Iteration:      8,   Func. Count:     41,   Neg. LLF: 10595.486065022078
Iteration:      9,   Func. Count:     45,   Neg. LLF: 10595.485652091902
Iteration:     10,   Func. Count:     49,   Neg. LLF: 10595.485650292876
Iteration:     11,   Func. Count:     52,   Neg. LLF: 10595.485650296207
Optimization terminated successfully    (Exit mode 0)
            Current function value: 10595.485650292876
            Iterations: 11
            Function evaluations: 52
            Gradient evaluat

In [ ]:
omega_garch_train_scaled = garch_result.params['omega']
alpha_garch_train = garch_result.params['alpha[1]']
beta_garch_train = garch_result.params['beta[1]']

***Rescaling***

In [ ]:
predicted_garch_volatility_train = predicted_garch_volatility_scaled_train/100
predicted_garch_variance_train = predicted_garch_variance_scaled_train/(100**2)
omega_garch_train = omega_garch_train_scaled/(100**2)

***Genuine OOS***

In [ ]:
test_array = np.array(test)

oos_residuals_expanding = []

for t in range(1, test_size + 1):
    expanding_returns = pd.concat([
        train_val,
        test.iloc[:t]
    ])
    cond_mean_exp = sm.tsa.statespace.SARIMAX(
        expanding_returns, order=(2,0,1), trend='c',
        enforce_stationarity=False, enforce_invertibility=False
    ).fit(disp=False, method='powell')

    # Last residual = residual of test[t-1]
    residual_t = cond_mean_exp.resid.iloc[-1]
    oos_residuals_expanding.append(residual_t)

oos_residuals_expanding = np.array(oos_residuals_expanding)

Streaming output truncated to the last 5000 lines.
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning:

A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.

/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning:

A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.

/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning:

A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.

/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning:

A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.

/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_mode

In [ ]:
oos_forecasts_garch = np.zeros(test_size) #sigma squared
oos_forecasts_garch[0] = omega_garch_train + alpha_garch_train*(np.array(train_val_residuals)[-1])**2 + (beta_garch_train * np.array(predicted_garch_variance_train)[-1])
oos_residuals_expanding_array = np.array(oos_residuals_expanding)
for t in range(1, test_size):
    oos_forecasts_garch[t] = omega_garch_train + alpha_garch_train * (oos_residuals_expanding_array[t-1]**2) + beta_garch_train * oos_forecasts_garch[t-1]

## Genuine Out-of-Sample Forecast: Winning Econometric Model

In [ ]:
test_residuals = residuals_df.iloc[train_size:].squeeze()

In [ ]:
fig_returns = go.Figure()
fig_returns.add_trace(go.Scatter(
    x=test.index, y=oos_forecasts_garch,
    mode='lines',
    name='OOS forecast',
    line=dict(color='firebrick', width=2.0)))

fig_returns.add_trace(go.Scatter(
    x=test.index, y=test_residuals**2,
    mode='lines',
    name='Squared Residuals',
    line=dict(color='steelblue', width=1.5),
    opacity=0.5))
fig_returns.update_layout(
    title=dict(
        text=f'GARCH(1,1) OOS Conditional Variance',
        x=0.5,
        font=dict(size=30, family='Serif', color='black')
    ),

    plot_bgcolor='rgba(0,0,0,0)',

    paper_bgcolor='rgba(0,0,0,0)',

    xaxis=dict(
        title='Date',
        titlefont=dict(size=25),
        tickfont=dict(size=25),
        showgrid=False
    ),
    yaxis=dict(
        title='Variance',
        titlefont=dict(size=25),
        tickfont=dict(size=25),
        showgrid=False,
        zeroline=False
    ),
    font=dict(
        family="Serif",
        size=25,
        color="black"
    ),
)

fig_returns.show()

## Out-of-Sample MAE: Winning Econometric Model

In [ ]:
OOS_MAE_winning_cond_var_model = np.mean(np.abs( test_residuals**2 - oos_forecasts_garch))
print(f"OOS_MAE_winning_cond_var_model:                        {OOS_MAE_winning_cond_var_model :.6f}")

OOS_MAE_winning_cond_var_model:                        0.000737


## 6. Machine Learning Benchmark: Gradient Boosting for Conditional Variance

In [ ]:
rv5 = pd.read_excel('/content/MktTbill_daily.xlsx', sheet_name='F-F_Research_Data_Factors_daily')
rv5.rename(columns={'Unnamed: 0': 'date'}, inplace=True)
rv5['date'] = pd.to_datetime(rv5['date'].astype(str), format='%Y%m%d')
rv5.set_index('date', inplace = True)
rv5 = rv5.loc['23/01/2001':'29/04/2025']
rv5.reset_index(inplace=True)

In [ ]:
rv5.head()

,date,Mkt-RF,RF,RV5
0,2001-01-23,1.57,0.026,0.147416
1,2001-01-24,0.34,0.026,0.161569
2,2001-01-25,-0.81,0.026,0.139099
3,2001-01-26,-0.11,0.026,0.079944
4,2001-01-29,0.90,0.026,0.116172


In [ ]:
df = rv5.rename(columns={"RV5": "rv5_vol"})
df["rv5_var"] = ((df["rv5_vol"] * 100) ** 2)/ 252

In [ ]:
df['squared_residuals'] = (residuals_df.reset_index()['Residuals']*100)**2

In [ ]:
df = df.drop(columns='rv5_vol')

 **Feature Engineering and Scaling**

We proceed to take the natural log of rv5 of the series seies of realised variance rv5_var as this will become our target variable.

In [ ]:
df['log_rv5_var'] = np.log(df['rv5_var'])
df['log_squared_residuals'] = np.log(df['squared_residuals'])

We now create calendar features.

In [ ]:
df['days_from_start'] = (df['date'] - df['date'].min()).dt.days
df['day_of_week']     = df['date'].dt.dayofweek          # 0 = Monday
df['day_of_month']    = df['date'].dt.day
df['week_of_year']    = df['date'].dt.isocalendar().week.astype(int)

Now we can create our numerical features relying on our log transformations.

In [ ]:
lags = [1, 5, 10, 22]
for lag in lags:
    df[f'lag_log_rv5_var_{lag}'] = (df['log_rv5_var']).shift(lag)
    df[f'lag_mkt_rf_{lag}'] = df['Mkt-RF'].shift(lag)
    df[f'lag_rf_{lag}']     = df['RF'].shift(lag)

We now drop the columns which are not lagged and do not coincide with our target variable.

In [ ]:
df.dropna(inplace=True)
df.reset_index(drop=True, inplace=True)

In [ ]:
df_rv5 = df[['date','squared_residuals','rv5_var','Mkt-RF','RF']]

In [ ]:
df = df.drop(columns=['rv5_var','Mkt-RF','RF','log_rv5_var','squared_residuals',])

**Separate the Target from the Features**

In [ ]:
drop_cols = ['date', 'log_squared_residuals']
X = [col for col in df.columns if col not in drop_cols]
X = df[X]

In [ ]:
X.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6081 entries, 0 to 6080
Data columns (total 16 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   days_from_start     6081 non-null   int64  
 1   day_of_week         6081 non-null   int32  
 2   day_of_month        6081 non-null   int32  
 3   week_of_year        6081 non-null   int64  
 4   lag_log_rv5_var_1   6081 non-null   float64
 5   lag_mkt_rf_1        6081 non-null   float64
 6   lag_rf_1            6081 non-null   float64
 7   lag_log_rv5_var_5   6081 non-null   float64
 8   lag_mkt_rf_5        6081 non-null   float64
 9   lag_rf_5            6081 non-null   float64
 10  lag_log_rv5_var_10  6081 non-null   float64
 11  lag_mkt_rf_10       6081 non-null   float64
 12  lag_rf_10           6081 non-null   float64
 13  lag_log_rv5_var_22  6081 non-null   float64
 14  lag_mkt_rf_22       6081 non-null   float64
 15  lag_rf_22           6081 non-null   float64
dtypes: flo

In [ ]:
y = df['log_squared_residuals']

## 7. Feature Engineering, Training, and Out-of-Sample Comparison vs. GARCH(1,1)

###**Split Train, Validation and Test**

In [ ]:
X_train_val, X_test, y_train_val, y_test = train_test_split(X, y, test_size=0.2, shuffle=False, random_state=42)

###**Global Scaling**

In [ ]:
feature_cols = X_train_val.columns.tolist()

Separate the calendar features from the numerical features

In [ ]:
calendar_cols = ['days_from_start', 'day_of_week',
                 'day_of_month', 'week_of_year']

numeric_cols  = [c for c in feature_cols if c not in calendar_cols]

Standardizing the numerical features

In [ ]:
train_val_means = X_train_val[numeric_cols].mean()
train_val_stds  = X_train_val[numeric_cols].std(ddof=0)
X_train_val_scaled = X_train_val.copy()
X_test_scaled      = X_test.copy()
X_train_val_scaled[numeric_cols] = (
    X_train_val[numeric_cols] - train_val_means
) / train_val_stds

X_test_scaled[numeric_cols] = (
    X_test[numeric_cols] - train_val_means
) / train_val_stds

Scale the target:

In [ ]:
y_train_val_mean = y_train_val.mean()
y_train_val_std  = y_train_val.std(ddof=0)

In [ ]:
y_train_val_scaled = y_train_val.copy()
y_test_scaled = y_test.copy()

In [ ]:
y_train_val_scaled = (y_train_val - y_train_val_mean) / y_train_val_std
y_test_scaled = (y_test - y_train_val_mean) / y_train_val_std

### **Cross-Validation for Time Series Data**

In [ ]:
tscv = TimeSeriesSplit(n_splits=5)

 ### **Model: Gradient Boosting**

In [ ]:
cond_var_model_ML_param_grid = {
    "max_depth"        : [3, 5, 10],
    "min_samples_split": [10, 50, 100],
    "n_estimators"     : [50, 100, 200],
    "learning_rate"    : [0.1, 0.05, 0.01, 0.005, 0.001]
}
cond_var_model_ML_results = []

In [ ]:
for max_depth in cond_var_model_ML_param_grid["max_depth"]:
  for min_samples_split in cond_var_model_ML_param_grid["min_samples_split"]:
    for n_estimators in cond_var_model_ML_param_grid["n_estimators"]:
      for learning_rate in cond_var_model_ML_param_grid["learning_rate"]:
        fold_mses = []
        for fold, (train_idx, val_idx) in enumerate(tscv.split(X_train_val), 1):

            # Prepare the various datasets
            X_train_fold, X_validation_fold = X_train_val.iloc[train_idx], X_train_val.iloc[val_idx]
            y_train_fold, y_validation_fold = y_train_val.iloc[train_idx], y_train_val.iloc[val_idx]

            # Scale correctly the features (numeric only)
            X_train_fold_mean = X_train_fold[numeric_cols].mean()
            X_train_fold_std  = X_train_fold[numeric_cols].std(ddof=0)
            X_train_fold_scaled = X_train_fold.copy()
            X_validation_fold_scaled = X_validation_fold.copy()
            X_train_fold_scaled[numeric_cols] = (X_train_fold[numeric_cols] - X_train_fold_mean) / X_train_fold_std
            X_validation_fold_scaled[numeric_cols] = (X_validation_fold[numeric_cols] - X_train_fold_mean) / X_train_fold_std

            # Scale correctly the target
            y_train_fold_mean = y_train_fold.mean()
            y_train_fold_std  = y_train_fold.std(ddof=0)
            y_train_fold_scaled = (y_train_fold - y_train_fold_mean) / y_train_fold_std
            y_validation_fold_scaled = (y_validation_fold - y_train_fold_mean) / y_train_fold_std

            # Train the model
            model = GradientBoostingRegressor(
                max_depth=max_depth,
                min_samples_split=min_samples_split,
                n_estimators=n_estimators,
                learning_rate=learning_rate,
                random_state=42
            )
            model.fit(X_train_fold_scaled, y_train_fold_scaled)

            # Predict and inverse-scale
            y_pred_scaled = model.predict(X_validation_fold_scaled)
            y_pred = y_pred_scaled * y_train_fold_std + y_train_fold_mean

            fold_mses.append(mean_squared_error(y_validation_fold, y_pred))

        avg_mse = np.mean(fold_mses)
        cond_var_model_ML_results.append({
            "model"                       : "GradientBoosting",
            "max_depth"                   : max_depth,
            "min_samples_split"           : min_samples_split,
            "n_estimators"                : n_estimators,
            "learning_rate"               : learning_rate,
            "average mse across 5 folds"  : avg_mse
        })

cond_var_model_ML_results_df = pd.DataFrame(cond_var_model_ML_results)
best_cond_var_model_ML = cond_var_model_ML_results_df.loc[cond_var_model_ML_results_df["average mse across 5 folds"].idxmin()]
print("\nBest GradientBoostingRegressor hyperparameters:")
print(best_cond_var_model_ML)


Best GradientBoostingRegressor hyperparameters:
model                         GradientBoosting
max_depth                                    3
min_samples_split                          100
n_estimators                               200
learning_rate                            0.005
average mse across 5 folds            5.816425
Name: 43, dtype: object


**Final Model**

In [ ]:
start_idx_test = X_test_scaled.index[0]
end_idx_test   = X_test_scaled.index[-1]
df_epsilon_squared_test = df_rv5.loc[start_idx_test : end_idx_test]

In [ ]:
# Gradient Boosting Final Model
final_cond_var_model_ML = GradientBoostingRegressor(
    max_depth=int(best_cond_var_model_ML['max_depth']),
    min_samples_split=int(best_cond_var_model_ML['min_samples_split']),
    n_estimators=int(best_cond_var_model_ML['n_estimators']),
    learning_rate=float(best_cond_var_model_ML['learning_rate']),
    random_state=42
)

final_cond_var_model_ML.fit(X_train_val_scaled, y_train_val_scaled)
y_pred_cond_var_model_ML_test_scaled = final_cond_var_model_ML.predict(X_test_scaled)
y_pred_cond_var_model_ML_test = y_pred_cond_var_model_ML_test_scaled * y_train_val_std + y_train_val_mean

In [ ]:
forecasts_final_cond_var_model_ML = pd.DataFrame(y_pred_cond_var_model_ML_test)

In [ ]:
forecasts_final_cond_var_model_ML.rename(columns={0: 'forecasts_final_cond_var_model_ML'}, inplace=True)
forecasts_final_cond_var_model_ML = forecasts_final_cond_var_model_ML.join(df_epsilon_squared_test.reset_index(drop=True))
forecasts_final_cond_var_model_ML['adjusted_forecasts_final_cond_var_model_ML'] = np.exp(
    forecasts_final_cond_var_model_ML["forecasts_final_cond_var_model_ML"]
    + 1/2 * np.var(forecasts_final_cond_var_model_ML["forecasts_final_cond_var_model_ML"])
)

In [ ]:
forecasts_final_cond_var_model_ML.set_index('date', inplace=True)

In [ ]:
forecast_errors_final_cond_var_model_ML = forecasts_final_cond_var_model_ML['adjusted_forecasts_final_cond_var_model_ML'] - forecasts_final_cond_var_model_ML['squared_residuals']
MAE_cond_var_model_ML = np.mean(np.abs(forecast_errors_final_cond_var_model_ML/(100**2)))
print(f"MAE_cond_var_model_ML:                        {MAE_cond_var_model_ML:.6f}")

MAE_cond_var_model_ML:                        0.000601


In [ ]:
MAE_cond_var_model_ML/OOS_MAE_winning_cond_var_model

np.float64(0.8155681786339504)

In [ ]:
fig_gb = go.Figure()

fig_gb.add_trace(go.Scatter(
    x=forecasts_final_cond_var_model_ML.index,
    y=forecasts_final_cond_var_model_ML['adjusted_forecasts_final_cond_var_model_ML']/(100**2),
    mode='lines',
    name='Adj Forecasts',
    line=dict(color='firebrick', width=3)
))

fig_gb.add_trace(go.Scatter(
    x=forecasts_final_cond_var_model_ML.index,
    y=forecasts_final_cond_var_model_ML['squared_residuals']/(100**2),
    mode='lines',
    name='squared_residuals',
    line=dict(color='steelblue', width=0.5)
))

fig_gb.update_layout(
    title=dict(
        text=f'Gradient Boosting Predicted Variance',
        x=0.5,
        font=dict(size=30, family='Serif', color='black')
    ),
    xaxis=dict(title='Date', titlefont=dict(size=25), tickfont=dict(size=20), showgrid=False),
    yaxis=dict(title='Variance', titlefont=dict(size=25), tickfont=dict(size=20), showgrid=False, zeroline=False),
    font=dict(family="Serif", size=20, color="black"),
    plot_bgcolor='rgba(0,0,0,0)',
    paper_bgcolor='rgba(0,0,0,0)',
    legend=dict(font=dict(size=18), bgcolor='rgba(255,255,255,0.5)', bordercolor='black', borderwidth=1),
    margin=dict(t=80, l=60, r=60, b=60)
)

fig_gb.show()

## Conclusion

- **Model selection is not just about fit.** EGARCH(1,1) and the Asymmetric Power ARCH model beat plain GARCH(1,1) on HQIC and log-likelihood, but only GARCH(1,1) passed the out-of-sample unbiasedness/efficiency regression test on its forecasts. In-sample fit statistics alone would have selected the wrong model for deployment.
- **Leverage effects matter.** The ARCH-LM test on the ARMA residuals revealed significant asymmetry, motivating the move to EGARCH; its estimated γ confirmed that negative shocks raise volatility more than positive ones of equal size.
- **Machine learning adds value, with caveats.** A Gradient Boosting model trained on market-wide realized variance, excess returns, and short rates achieved a lower out-of-sample MAE than GARCH(1,1) (0.000601 vs 0.000737), despite lacking GARCH's direct access to the stock's own lagged residual. However, its forecasts are much smoother and largely miss volatility spikes — a reminder that MAE, dominated by calm periods, can mask poor tail-risk performance.
- **Next steps:** evaluate with a loss function that penalizes large errors more heavily (e.g. MSE or QLIKE), add VIX as a forward-looking volatility feature, and test whether combining the econometric and ML forecasts improves on either alone.
